#**Training a Song Embedding Model**
Training a song embedding model using Word2Vec, where playlists are treated like sentences and songs are treated like words.





In [1]:
import pandas as pd
from urllib import request

In [2]:
# Download the playlist dataset
data = request.urlopen(
    "https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt"
)

In [3]:
# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split("\n")[2:]

In [4]:
# Remove playlists containing only one song
# It is discarded because Word2Vec needs surrounding context to learn relationships.
playlists = [
    s.rstrip().split()
    for s in lines
    if len(s.split()) > 1
]

In [5]:
# Download song metadata
# This file maps song IDs to human-readable information such as titles and artists.
songs_file = request.urlopen(
    "https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt"
)

In [ ]:
# Parse the song metadata
songs_file = songs_file.read().decode("utf-8").split("\n")

songs = [
    s.rstrip().split("\t")
    for s in songs_file
]

In [24]:
# Create a DataFrame
songs_df = pd.DataFrame(
    data=songs,
    columns=["id", "title", "artist"]
)

In [25]:
print(songs_df.columns)

Index(['id', 'title', 'artist'], dtype='object')


In [26]:
print(songs_df.index)

RangeIndex(start=0, stop=75262, step=1)


In [27]:
# Set the song ID as the index
songs_df = songs_df.set_index("id")

In [28]:
print(songs_df.head())

                                                title      artist
id                                                               
0                        Gucci Time (w\/ Swizz Beatz)  Gucci Mane
1   Aston Martin Music (w\/ Drake & Chrisette Mich...   Rick Ross
2                       Get Back Up (w\/ Chris Brown)        T.I.
3                  Hot Toddy (w\/ Jay-Z & Ester Dean)       Usher
4                                        Whip My Hair      Willow


In [29]:
# Inspect playlists
print("Playlist #1:", playlists[0])
print("Playlist #2:", playlists[1])

Playlist #1: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43']
Playlist #2: ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117', '118'

##Train a Word2Vec model

In [ ]:
# The model learns that words appearing together are related.
# Word2Vec is based on the intuition that items appearing in similar contexts should have similar representations.

# In text: words that occur in similar sentences have similar embeddings.
# Here: songs that occur in similar playlists have similar embeddings.

In [15]:
!pip install gensim
from gensim.models import Word2Vec

model = Word2Vec(
    playlists,
    vector_size=32,
    window=20,
    negative=50,
    min_count=1,
    workers=4
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 79.2 MB/s eta 0:00:00


In [30]:
song_id = 2172
# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('2976', 0.9975847601890564),
 ('5549', 0.996795117855072),
 ('3114', 0.9967671036720276),
 ('6624', 0.9965450763702393),
 ('2886', 0.9962307810783386),
 ('3099', 0.9958100914955139),
 ('2849', 0.9957128167152405),
 ('2640', 0.9945201277732849),
 ('3136', 0.9943153262138367),
 ('6640', 0.9942582249641418)]

In [31]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172, dtype: object


In [32]:
import numpy as np

def print_recommendations(song_id):
    similar_ids = [
        song_id
        for song_id, _ in model.wv.most_similar(
            positive=str(song_id), topn=5
        )
    ]
    return songs_df.loc[similar_ids]

print(print_recommendations(2172))

                         title         artist
id                                           
2976              I Don't Know  Ozzy Osbourne
5549             November Rain  Guns N' Roses
3114  And The Cradle Will Rock      Van Halen
6624   Everybody Wants Some!!!      Van Halen
2886                   The Zoo      Scorpions


#END